# 122 — Eval CI Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/122-eval-ci-pipeline/eval_ci_pipeline_workbook.ipynb)

**Concept:** Wire LLM evaluation metrics as pytest tests. A score regression fails the CI build automatically — no human review needed.

**What you'll build:**
- A 10-item golden dataset for a Q&A pipeline
- A good pipeline (uses context) and a degraded pipeline (ignores context)
- Local scoring functions for faithfulness and answer relevancy
- The pytest integration pattern using DeepEval
- A GitHub Actions CI config that gates merges on eval scores

**Why this matters:** LLM pipelines degrade silently. Without eval CI, a bad prompt change ships to production undetected. With eval CI, the test suite catches regressions in seconds.

## Setup

Install dependencies. `deepeval` is optional — the workbook uses a local fallback scoring system by default. If you have a `DEEPEVAL_API_KEY`, you can swap in real DeepEval metrics later.

In [ ]:
!pip install openai python-dotenv -q
# Optional: uncomment to install DeepEval (requires DEEPEVAL_API_KEY)
# !pip install deepeval -q

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# os.environ["DEEPEVAL_API_KEY"] = userdata.get("DEEPEVAL_API_KEY")  # optional

## Part 1 — Why Eval CI? The Problem with Manual LLM Testing

Traditional software has deterministic outputs — unit tests catch regressions reliably. LLMs are different:

- The same prompt can produce different outputs on different days
- A small wording change in your system prompt can degrade quality across 1000 queries
- Manual spot-checking misses regressions that only appear in edge cases

**The solution:** Build a golden dataset of representative inputs and expected outputs. Run the pipeline against it on every commit. Assert that quality scores stay above a threshold. If they don't — the CI build fails.

This is exactly how production ML teams at OpenAI, Anthropic, and Google operate their LLM pipelines.

## Part 2 — Golden Datasets: What They Are and How to Build Them

A **golden dataset** is a curated set of (input, expected_output, context) triples that represent your pipeline's core use cases.

**Rules for a good golden dataset:**
1. Cover the most common queries your users actually ask
2. Cover edge cases where the pipeline is likely to fail
3. Include the expected output (what a human expert would say)
4. Include the retrieval context (what your RAG system would return)
5. Keep it small enough to run in under 60 seconds (10-50 items is ideal)

Let's look at our 10-item golden dataset:

In [ ]:
MODEL = "gpt-4o-mini"

GOLDEN_DATASET = [
    {
        "question": "What is a LangGraph node?",
        "context": "In LangGraph, a node is a Python function that receives the current state dict and returns an updated state dict. Nodes are the processing units in a LangGraph workflow.",
        "expected_output": "A LangGraph node is a Python function that receives and returns a state dict.",
    },
    {
        "question": "What does RAG stand for?",
        "context": "RAG stands for Retrieval-Augmented Generation. It combines a retrieval system (usually vector search) with a generative LLM to produce grounded, factual responses.",
        "expected_output": "RAG stands for Retrieval-Augmented Generation.",
    },
    {
        "question": "What is the purpose of temperature in LLM calls?",
        "context": "Temperature controls the randomness of LLM outputs. A temperature of 0 produces deterministic, focused outputs. Higher temperatures (0.7-1.0) produce more creative and varied outputs.",
        "expected_output": "Temperature controls randomness; 0 is deterministic, higher values increase creativity.",
    },
    {
        "question": "What is prompt injection?",
        "context": "Prompt injection is an attack where malicious content in user input or retrieved documents overrides the LLM's original instructions. It is a key security risk in agentic systems.",
        "expected_output": "Prompt injection overrides LLM instructions via malicious user input or retrieved content.",
    },
    {
        "question": "What is the difference between gpt-4o and gpt-4o-mini?",
        "context": "gpt-4o is OpenAI's flagship multimodal model with highest capability. gpt-4o-mini is a smaller, faster, and cheaper version optimized for cost-efficiency while maintaining strong performance on most tasks.",
        "expected_output": "gpt-4o is the flagship model; gpt-4o-mini is smaller, faster, and cheaper.",
    },
    {
        "question": "What is a vector database?",
        "context": "A vector database stores high-dimensional embeddings and supports similarity search (nearest neighbor). Examples include Chroma, Pinecone, and Weaviate. They are the retrieval backbone of most RAG systems.",
        "expected_output": "A vector database stores embeddings and enables similarity search for RAG retrieval.",
    },
    {
        "question": "What is tool calling in LLMs?",
        "context": "Tool calling (also called function calling) allows LLMs to request the execution of defined functions. The LLM outputs a structured JSON call, which the framework executes and returns results back to the LLM.",
        "expected_output": "Tool calling lets LLMs request function execution by outputting structured JSON calls.",
    },
    {
        "question": "What is the system prompt?",
        "context": "The system prompt is an instruction given to an LLM at the start of a conversation, before any user messages. It sets the AI's persona, task, constraints, and context.",
        "expected_output": "The system prompt sets the AI's persona, task, and constraints before conversation starts.",
    },
    {
        "question": "What is chain-of-thought prompting?",
        "context": "Chain-of-thought (CoT) prompting encourages LLMs to show their reasoning step by step before giving a final answer. It was shown to significantly improve performance on multi-step reasoning tasks.",
        "expected_output": "CoT prompting makes LLMs show step-by-step reasoning, improving multi-step task performance.",
    },
    {
        "question": "What is a hallucination in LLM outputs?",
        "context": "A hallucination is when an LLM generates factually incorrect or fabricated information presented as true. Hallucinations are more common when the model lacks context or is asked about obscure facts.",
        "expected_output": "A hallucination is when an LLM generates factually incorrect or fabricated information.",
    },
]

print(f"Golden dataset: {len(GOLDEN_DATASET)} Q&A pairs")
for i, row in enumerate(GOLDEN_DATASET[:3], 1):
    print(f"\n[{i}] Q: {row['question']}")
    print(f"    Expected: {row['expected_output']}")

## Part 3 — The Pipeline Under Test

We'll compare two pipelines:

1. **Good pipeline**: Uses the provided context to answer. Should score high.
2. **Degraded pipeline**: Ignores context, returns a generic non-answer. Should fail CI.

This mirrors a real scenario: you ship a prompt change that accidentally removes the context injection, and now your RAG pipeline answers everything with "I don't know."

In [ ]:
from openai import OpenAI

client = OpenAI()


def run_pipeline(question: str, context: str) -> str:
    """Good pipeline: answers using the provided context."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": f"Answer the question using ONLY the provided context. Be concise.\n\nContext: {context}"},
            {"role": "user", "content": question},
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()


def degraded_pipeline(question: str, context: str) -> str:
    """Bad pipeline: ignores context entirely."""
    return "I don't know. Please consult relevant documentation for more information."

In [ ]:
# Compare both pipelines on a sample question
sample = GOLDEN_DATASET[0]

good_answer = run_pipeline(sample["question"], sample["context"])
bad_answer = degraded_pipeline(sample["question"], sample["context"])

print(f"Question: {sample['question']}")
print(f"\nExpected: {sample['expected_output']}")
print(f"\nGood pipeline: {good_answer}")
print(f"\nDegraded pipeline: {bad_answer}")

## Part 4 — Evaluation Metrics

We use two metrics from DeepEval's standard suite:

**Faithfulness** — Does the answer stick to the retrieved context? High faithfulness means the model isn't hallucinating. Low faithfulness means the model is ignoring the context or making things up.

**Answer Relevancy** — Does the answer actually address the question? High relevancy means the response is on-topic. Low relevancy means the model is giving a generic or off-topic answer.

For this workbook, we use a simple local proxy (keyword overlap) instead of requiring the DeepEval API. In production, swap in real DeepEval metrics for LLM-graded evaluation.

In [ ]:
def score_faithfulness_simple(actual: str, context: str) -> float:
    """Local proxy: fraction of context keywords found in the answer."""
    context_words = set(context.lower().split())
    actual_words = set(actual.lower().split())
    common = context_words & actual_words
    return round(min(len(common) / max(len(context_words) * 0.15, 1), 1.0), 3)


def score_relevancy_simple(actual: str, question: str) -> float:
    """Local proxy: fraction of question keywords found in the answer."""
    stop = {"what", "is", "the", "a", "an", "of", "in", "does"}
    q_words = set(question.lower().split()) - stop
    actual_words = set(actual.lower().split())
    if not q_words:
        return 1.0
    return round(len(q_words & actual_words) / len(q_words), 3)


# Test on sample
f_good = score_faithfulness_simple(good_answer, sample["context"])
r_good = score_relevancy_simple(good_answer, sample["question"])
f_bad = score_faithfulness_simple(bad_answer, sample["context"])
r_bad = score_relevancy_simple(bad_answer, sample["question"])

print("Good pipeline scores:")
print(f"  Faithfulness:  {f_good}")
print(f"  Relevancy:     {r_good}")
print("\nDegraded pipeline scores:")
print(f"  Faithfulness:  {f_bad}")
print(f"  Relevancy:     {r_bad}")

In [ ]:
def build_test_case(row: dict, actual_output: str) -> dict:
    """Build a DeepEval-compatible test case dict."""
    return {
        "input": row["question"],
        "actual_output": actual_output,
        "expected_output": row["expected_output"],
        "context": [row["context"]],
    }


PASS_THRESHOLD = 0.5


def evaluate_pipeline(pipeline_fn, dataset, label):
    """Run pipeline on full dataset and report CI pass/fail."""
    print(f"\n--- {label} ---")
    f_scores, r_scores = [], []
    for row in dataset:
        actual = pipeline_fn(row["question"], row["context"])
        f_scores.append(score_faithfulness_simple(actual, row["context"]))
        r_scores.append(score_relevancy_simple(actual, row["question"]))
    avg_f = sum(f_scores) / len(f_scores)
    avg_r = sum(r_scores) / len(r_scores)
    avg = (avg_f + avg_r) / 2
    passed = avg >= PASS_THRESHOLD
    print(f"  Faithfulness:    {avg_f:.3f}")
    print(f"  Answer Relevancy:{avg_r:.3f}")
    print(f"  Combined:        {avg:.3f}")
    print(f"  CI status:       {'PASS' if passed else 'FAIL'}")
    return {"avg_combined": avg, "passed": passed}


good_result = evaluate_pipeline(run_pipeline, GOLDEN_DATASET, "Good Pipeline")
bad_result = evaluate_pipeline(degraded_pipeline, GOLDEN_DATASET, "Degraded Pipeline")

## Part 5 — The pytest Integration Pattern

The real power of eval CI comes from integrating scores with pytest. When a test fails, the CI build fails — no special CI logic needed.

With DeepEval, each test case is an `LLMTestCase` and each metric is a `FaithfulnessMetric` or `AnswerRelevancyMetric`. The `assert_test()` call handles the threshold check.

In [ ]:
PYTEST_PATTERN = '''
# tests/test_pipeline_eval.py
import pytest
from deepeval import assert_test
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

# Import your pipeline and dataset
from src.tools import run_pipeline, load_golden_dataset


@pytest.mark.parametrize("row", load_golden_dataset())
def test_pipeline_faithfulness(row):
    actual = run_pipeline(row["question"], row["context"])
    test_case = LLMTestCase(
        input=row["question"],
        actual_output=actual,
        expected_output=row["expected_output"],
        retrieval_context=[row["context"]]
    )
    assert_test(test_case, [
        FaithfulnessMetric(threshold=0.5),
        AnswerRelevancyMetric(threshold=0.5),
    ])
'''

print(PYTEST_PATTERN)

## Part 6 — GitHub Actions CI Config

Once you have pytest tests, adding them to CI is just a workflow file. The key: run `pytest tests/test_pipeline_eval.py` as a required check. If it fails, the PR cannot merge.

In [ ]:
GH_ACTIONS_CONFIG = '''
# .github/workflows/eval-ci.yml
name: Eval CI

on:
  pull_request:
    branches: [main]
  push:
    branches: [main]

jobs:
  eval:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: pip install openai deepeval python-dotenv pytest

      - name: Run eval CI tests
        env:
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
          DEEPEVAL_API_KEY: ${{ secrets.DEEPEVAL_API_KEY }}
        run: pytest tests/test_pipeline_eval.py -v
'''

print(GH_ACTIONS_CONFIG)

## Part 7 — Score Thresholds and Regression Detection

The threshold determines how strict your CI is. Too low and regressions slip through. Too high and your CI becomes flaky.

**Practical starting points:**
- 0.5 — lenient, catches obvious regressions only
- 0.7 — moderate, catches most quality drops
- 0.85+ — strict, suitable for production-critical pipelines

Let's see what happens when we raise the threshold:

In [ ]:
def check_threshold(result_score: float, threshold: float, label: str):
    passed = result_score >= threshold
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {label}: score={result_score:.3f}, threshold={threshold}")


good_score = good_result["avg_combined"]
bad_score = bad_result["avg_combined"]

print("Threshold impact on CI results:")
for threshold in [0.3, 0.5, 0.7, 0.85]:
    print(f"\nThreshold = {threshold}:")
    check_threshold(good_score, threshold, "Good pipeline")
    check_threshold(bad_score, threshold, "Degraded pipeline")

## Exercises

Practice by extending the example:

**Exercise 1:** Add a new Q&A pair to the golden dataset about a topic of your choice. Include a question, context paragraph, and expected output. Then re-run the evaluation.

**Exercise 2:** Write a partially-degraded pipeline that uses context sometimes but not always (e.g., only uses context if the question contains the word "what"). Evaluate it and see where it lands between the good and degraded scores.

In [ ]:
# Exercise 1: Add a new Q&A pair
# YOUR CODE HERE
new_pair = {
    "question": "YOUR QUESTION HERE",
    "context": "YOUR CONTEXT HERE",
    "expected_output": "YOUR EXPECTED OUTPUT HERE",
}

# Extend the dataset and re-run
extended_dataset = GOLDEN_DATASET + [new_pair]
print(f"Extended dataset size: {len(extended_dataset)}")

In [ ]:
# Exercise 2: Write a partially-degraded pipeline
# YOUR CODE HERE
def partial_pipeline(question: str, context: str) -> str:
    """Uses context only when question contains 'what'."""
    if "what" in question.lower():
        return run_pipeline(question, context)
    return "I don't know. Please consult relevant documentation for more information."


partial_result = evaluate_pipeline(partial_pipeline, GOLDEN_DATASET, "Partial Pipeline")
print(f"\nFinal comparison:")
print(f"  Good pipeline:    {good_score:.3f}")
print(f"  Partial pipeline: {partial_result['avg_combined']:.3f}")
print(f"  Degraded pipeline:{bad_score:.3f}")

## Answer Key

**Exercise 1 — Sample answer:**

In [ ]:
# Answer Key — Exercise 1
new_pair_answer = {
    "question": "What is few-shot prompting?",
    "context": "Few-shot prompting provides 2-5 examples of the desired input-output format directly in the prompt. This helps the LLM understand the task pattern without fine-tuning, and is especially effective for formatting and classification tasks.",
    "expected_output": "Few-shot prompting includes 2-5 examples in the prompt to teach the LLM the desired output format.",
}

extended_dataset = GOLDEN_DATASET + [new_pair_answer]
result = evaluate_pipeline(run_pipeline, extended_dataset, "Extended Dataset")
print(f"\nDataset now has {len(extended_dataset)} items — CI still passes: {result['passed']}")

## Workshop Complete

**What you learned:**
- Golden datasets capture the expected behavior of your pipeline
- Faithfulness and Answer Relevancy are the two core RAG eval metrics
- Local keyword-overlap scoring is a fast, free approximation
- DeepEval provides LLM-graded metrics for production use
- Wiring eval as pytest tests makes CI block bad deployments automatically

**Next steps:**
1. Install DeepEval and swap in real metrics for your pipeline
2. Add `pytest tests/test_pipeline_eval.py` to your CI workflow
3. Set your threshold based on your current baseline score
4. Expand your golden dataset to cover more edge cases

**The insight:** Once eval CI is set up, every PR that degrades quality fails automatically. You've turned a manual, error-prone process into a deterministic gate.